# ROCKET Baseline — NFC Relay Detection (random split vs. Leave-One-Tag-Out)

Baseline for the NFC ATQA dataset produced by `wav_to_tabular.py`. Reads the per-label split CSVs
(`nfc_atqa_tag1_normal.csv` … `nfc_atqa_wireless_relay.csv`) or a single combined CSV. Same ROCKET-family
pipeline as the KeFRA notebook — MiniRocket features + a ridge classifier, evaluated with out-of-fold
predictions and read out as a `classification_report` plus confusion matrix.

**Why this notebook runs two evaluations**

| Evaluation | CNN accuracy |
|---|---|
| Random / standard split | ~99.99% |
| Leave-One-Tag-Out (LOTO) | ~65% (wired-relay F1 ≈ 0 on 3 of 4 held-out tags) |

A random split lets samples from the *same physical tag* land in both train and test, so the model can win
by memorizing each tag's hardware fingerprint instead of learning what a relay does. LOTO removes that
shortcut: train on 3 tags, test on the 4th unseen tag. The gap between the two is the result.

**The KeFRA connection.** In the KeFRA notebook we flagged that a high score deserved a caveat because the
dataset had no grouping column, so we *couldn't* rule out session leakage. NFC is the opposite — the
`physical_tag` column lets us build exactly that leakage-free grouped split. LOTO is the honest evaluation
KeFRA wished it had.

> Like KeFRA, this is supervised classification (it trains on labeled attacks), not one-class anomaly
> detection — and it learns from extracted ATQA segments, not live RF. Treat it as a methodology baseline
> against the CNN, not a deployable detector.

## 1. Setup

In [ ]:
# Colab: %pip install aeon scikit-learn numpy pandas matplotlib
import glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import RidgeClassifierCV
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

USE_MINIROCKET = True
if USE_MINIROCKET:
    from aeon.transformations.collection.convolution_based import MiniRocket as RocketTransform
else:
    from aeon.transformations.collection.convolution_based import Rocket as RocketTransform
print("Transform:", RocketTransform.__name__)

## 2. Config

`DATA_GLOB` matches your split files. It also works if you point it at a single combined CSV.

In [ ]:
DATA_GLOB      = "data/nfc_atqa_*.csv"   # split files; or "data/nfc_atqa.csv" for the combined file
CLASS_COLUMN   = "class"                  # 3-class target: normal / wired_relay / wireless_relay
TAG_COLUMN     = "physical_tag"           # tag1..tag4  (used for LOTO)
INDEX_COLUMN   = "sample_index"           # used to spread wireless_relay across LOTO folds

CLASSES        = ["normal", "wired_relay", "wireless_relay"]
NORMAL_CLASSES = ["normal"]               # everything else is an attack (binary view)
TAGS           = ["tag1", "tag2", "tag3", "tag4"]
N_RANDOM_FOLDS = 5
RANDOM_STATE   = 42
MAX_PER_CLASS  = None     # set e.g. 4000 if RAM is tight; None = use everything

## 3. Load the split CSVs

Each file carries the metadata columns plus `x0…x1799`. We concatenate them; the signal is the `x*`
columns, the target is `class`, and `physical_tag` / `sample_index` drive the LOTO split.

In [ ]:
files = sorted(glob.glob(DATA_GLOB))
assert files, f"No files matched {DATA_GLOB}"
print(f"Loading {len(files)} file(s):")
for f in files: print("  ", os.path.basename(f))

df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

if MAX_PER_CLASS:
    df = (df.groupby(CLASS_COLUMN, group_keys=False)
            .apply(lambda g: g.sample(min(len(g), MAX_PER_CLASS), random_state=RANDOM_STATE)))

sig_cols = [c for c in df.columns if c.startswith("x") and c[1:].isdigit()]
sig_cols = sorted(sig_cols, key=lambda c: int(c[1:]))
print(f"\n{len(df)} samples, {len(sig_cols)} signal points each")
print(df[CLASS_COLUMN].value_counts())

X    = df[sig_cols].to_numpy(np.float32)[:, np.newaxis, :]   # (N, 1, 1800)
y    = df[CLASS_COLUMN].to_numpy()
tags = df[TAG_COLUMN].to_numpy()
sidx = df[INDEX_COLUMN].to_numpy()

## 4. Sanity check — one waveform per class

If the three classes look at least subtly different, the WAV→CSV conversion preserved the signal. If they
are flat or identical, fix that upstream before trusting any accuracy.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for cls in CLASSES:
    i = np.where(y == cls)[0][0]
    ax.plot(X[i, 0], lw=0.8, label=cls)
ax.set_title("One ATQA waveform per class"); ax.set_xlabel("sample point"); ax.legend()
plt.tight_layout(); plt.show()

## 5. Shared helpers

`fit_predict` fits the kernels on the training fold only, standardizes features with training statistics,
then fits a ridge classifier — the kernels never see test data. `run_oof` runs a list of splits and
collects out-of-fold predictions (every sample predicted exactly once, by a model that didn't train on it).
`report_oof` prints the KeFRA-style accuracy + `classification_report` and plots the confusion matrix.

In [ ]:
def fit_predict(Xtr, ytr, Xte):
    rk = RocketTransform(random_state=RANDOM_STATE)
    Ftr = np.asarray(rk.fit_transform(Xtr))
    Fte = np.asarray(rk.transform(Xte))
    mu, sd = Ftr.mean(0), Ftr.std(0) + 1e-8
    clf = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
    clf.fit((Ftr - mu) / sd, ytr)
    return clf.predict((Fte - mu) / sd)

def run_oof(splits, X, y):
    """splits: list of (name, train_idx, test_idx). Returns (oof_pred, fold_id, per_fold_acc)."""
    oof     = np.empty(len(y), dtype=object)
    fold_id = np.empty(len(y), dtype=object)
    accs    = []
    for name, tr, te in splits:
        pred = fit_predict(X[tr], y[tr], X[te])
        oof[te], fold_id[te] = pred, name
        a = accuracy_score(y[te], pred); accs.append(a)
        print(f"{name}: acc = {a*100:.2f}%")
    print(f"\nMean accuracy: {np.mean(accs)*100:.2f}% (+/- {np.std(accs)*100:.2f})")
    return oof, fold_id, np.array(accs)

def report_oof(title, y, oof, labels=CLASSES):
    acc = accuracy_score(y, oof)
    print(f"OOF accuracy: {acc*100:.2f}%\n")
    print(classification_report(y, oof, labels=labels, digits=4, zero_division=0))
    cm = confusion_matrix(y, oof, labels=labels)
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(cm, display_labels=labels).plot(ax=ax, xticks_rotation=45, colorbar=True)
    ax.set_title(f"{title} (OOF acc {acc*100:.2f}%)"); plt.tight_layout(); plt.show()
    return acc, cm

def per_fold_table(y, oof, fold_id, fold_names):
    """Per-fold rows in the same columns as loto_cnn_results.csv."""
    rows = []
    for name in fold_names:
        m = fold_id == name
        f1s = f1_score(y[m], oof[m], labels=CLASSES, average=None, zero_division=0)
        rows.append({"fold": name,
                     "accuracy": accuracy_score(y[m], oof[m]),
                     "macro_f1": f1_score(y[m], oof[m], labels=CLASSES, average="macro", zero_division=0),
                     "normal_f1": f1s[0], "wired_relay_f1": f1s[1], "wireless_relay_f1": f1s[2]})
    return pd.DataFrame(rows)

## 6. Evaluation A — random stratified split

The optimistic setting that matches the paper and the CNN's ~99.99%. The same physical tags appear in train
and test, so a high score here is expected — and is exactly the number LOTO will deflate.

In [ ]:
skf = StratifiedKFold(n_splits=N_RANDOM_FOLDS, shuffle=True, random_state=RANDOM_STATE)
rand_splits = [(f"fold {k}", tr, te) for k, (tr, te) in enumerate(skf.split(np.zeros(len(y)), y), 1)]

rand_oof, rand_fold, rand_accs = run_oof(rand_splits, X, y)

In [ ]:
report_oof("ROCKET random split — 3-class", y, rand_oof)

## 7. Evaluation B — Leave-One-Tag-Out (the honest test)

For each held-out tag, **all** of that tag's `normal` and `wired_relay` samples go to test; the model trains
on the other three tags. `wireless_relay` comes from a single emulator (not one of the 4 tags), so it can't
be held out by tag — we spread it deterministically across folds via `sample_index % 4`, so it appears in
both train and test in every fold. This isolates the real question: *does relay detection generalize to a
physical tag the model has never seen?*

In [ ]:
fold_key = np.where(
    np.isin(y, ["normal", "wired_relay"]),
    tags,
    np.array([f"tag{(int(s) % 4) + 1}" for s in sidx])
)
loto_splits = [(t, np.where(fold_key != t)[0], np.where(fold_key == t)[0]) for t in TAGS]

loto_oof, loto_fold, loto_accs = run_oof(loto_splits, X, y)

In [ ]:
report_oof("ROCKET LOTO — 3-class", y, loto_oof)

# Security-critical number: relays accepted as normal
cm = confusion_matrix(y, loto_oof, labels=CLASSES)
relay_as_normal = (cm[CLASSES.index("wired_relay"),    CLASSES.index("normal")]
                 + cm[CLASSES.index("wireless_relay"), CLASSES.index("normal")])
print(f"Relay samples accepted as NORMAL under LOTO: {relay_as_normal}")

## 8. Binary view — normal vs. relay (under LOTO)

Collapses both relay types into one attack class — the detection-oriented framing, and the KeFRA-style
real-vs-fake collapse. Watch whether the errors that survive are relay↔normal (dangerous) or just
wired↔wireless confusions (harmless for detection).

In [ ]:
to_bin = lambda a: np.where(np.isin(a, NORMAL_CLASSES), "normal", "relay")
ytb, ypb = to_bin(y), to_bin(loto_oof)

print(f"Binary OOF accuracy: {accuracy_score(ytb, ypb)*100:.2f}%\n")
print(classification_report(ytb, ypb, labels=["normal", "relay"], digits=4, zero_division=0))

## 9. Save results + compare to CNN

Writes per-fold tables in the same columns as `loto_cnn_results.csv` so you can stack the two models.

In [ ]:
rand_table = per_fold_table(y, rand_oof, rand_fold, [f"fold {k}" for k in range(1, N_RANDOM_FOLDS + 1)])
loto_table = per_fold_table(y, loto_oof, loto_fold, TAGS)

rand_table.to_csv("rocket_random_results.csv", index=False)
loto_table.to_csv("rocket_loto_results.csv", index=False)
print("Saved rocket_random_results.csv and rocket_loto_results.csv\n")

print("ROCKET-family summary")
print(f"  Random split : {rand_accs.mean()*100:.2f}%")
print(f"  LOTO         : {loto_accs.mean()*100:.2f}%\n")
print("Compare per-fold to the CNN (if loto_cnn_results.csv is alongside):")
print("  cnn = pd.read_csv('loto_cnn_results.csv')")
print("  pd.merge(cnn, loto_table, on='fold', suffixes=('_cnn','_rocket'))")
loto_table

## 10. Notes

- **The headline is the gap, not the accuracy.** A high random-split score next to a low LOTO score is the
  result: it shows the model keys on tag identity, not relay behavior. Report both numbers together.
- **Watch `wired_relay_f1` per fold.** If it craters on held-out tags (as in the CNN run), the detector
  can't recognize a relayed signal from a tag it never trained on — the realistic deployment case.
- **Where do the errors live?** Same lens as KeFRA: a wired↔wireless mix-up is harmless for detection;
  a relay→normal mistake is an attack passing as legitimate. The binary view and the relay-accepted-as-normal
  count tell you which kind you have.
- **LOTO is the leakage-free split KeFRA couldn't do.** Because NFC exposes `physical_tag`, this notebook
  delivers the grouped evaluation directly — the strongest evidence in the project that the inflated
  random-split accuracy is an artifact, not real generalization.
- **Two models, one conclusion.** ROCKET and the CNN reaching the same random-vs-LOTO pattern is stronger
  than either alone — it pins the effect on the data/evaluation, not one architecture.